# Task 4: TF-IDF Vectorization and Similarity Analysis

## Objective
Convert the combined text of TED Talks into numerical TF-IDF
vectors and use cosine similarity to identify related talks.

## Activities
1. Load and inspect the cleaned dataset.
2. Apply TF-IDF vectorization.
3. Inspect the vocabulary and matrix dimensions.
4. Calculate cosine similarity.
5. Generate the top five recommendations for three talks.
6. Review the recommendations and document limitations.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\processed\ted_talks_cleaned.csv")

In [3]:
df.head(3)

,title,description,tags,main_speaker,speaker_occupation,event,views,url,transcript
0,do schools kill creativity?,sir ken robinson makes an entertaining and pro...,children creativity culture dance education pa...,Ken Robinson,Author/educator,TED2006,47227110,https://www.ted.com/talks/ken_robinson_says_sc...,"good morning. how are you? it's been great, ha..."
1,averting the climate crisis,with the same humor and humanity he exuded in ...,alternative energy cars climate change culture...,Al Gore,Climate advocate,TED2006,3200520,https://www.ted.com/talks/al_gore_on_averting_...,"thank you so much, chris. and it's truly a gre..."
2,simplicity sells,new york times columnist david pogue takes aim...,computers entertainment interface design media...,David Pogue,Technology columnist,TED2006,1636292,https://www.ted.com/talks/david_pogue_says_sim...,"hello voice mail, my old friend. i've called f..."


In [4]:
df.shape

(2550, 9)

In [5]:
print("Available columns:")
print(df.columns.tolist())

Available columns:
['title', 'description', 'tags', 'main_speaker', 'speaker_occupation', 'event', 'views', 'url', 'transcript']


In [6]:
# Check required columns
required_columns = ["title", "description", "tags", "transcript"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Check missing values
print("\nMissing values:")
print(df[required_columns].isnull().sum())

# Check total records
print("\nTotal records:", len(df))

# Check missing titles
print("Missing titles:", df["title"].isna().sum())


Missing values:
title           0
description     0
tags            0
transcript     94
dtype: int64

Total records: 2550
Missing titles: 0


In [7]:
# Replace missing text with empty strings
text_columns = ["title", "tags", "description", "transcript"]

for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

# Combine the text fields into one column
df["combined_text"] = (
    df["title"] + " " +
    df["tags"] + " " +
    df["description"] + " " +
    df["transcript"]
).str.strip()

# Display the result
display(df[["title", "combined_text"]].head())

,title,combined_text
0,do schools kill creativity?,do schools kill creativity? children creativit...
1,averting the climate crisis,averting the climate crisis alternative energy...
2,simplicity sells,simplicity sells computers entertainment inter...
3,greening the ghetto,greening the ghetto macarthur grant activism b...
4,the best stats you've ever seen,the best stats you've ever seen africa asia go...


In [8]:
# Identify talks with non-empty combined text
usable_mask = df["combined_text"].str.strip().ne("")

print("Total records:", len(df))
print("Usable records:", usable_mask.sum())
print("Empty text records:", (~usable_mask).sum())

# Keep only usable records
talks = df.loc[usable_mask].copy().reset_index(drop=True)

print("\nUsable dataset shape:", talks.shape)

display(talks[["title", "combined_text"]].head())

Total records: 2550
Usable records: 2550
Empty text records: 0

Usable dataset shape: (2550, 10)


,title,combined_text
0,do schools kill creativity?,do schools kill creativity? children creativit...
1,averting the climate crisis,averting the climate crisis alternative energy...
2,simplicity sells,simplicity sells computers entertainment inter...
3,greening the ghetto,greening the ghetto macarthur grant activism b...
4,the best stats you've ever seen,the best stats you've ever seen africa asia go...


In [9]:
documents = talks["combined_text"].tolist()

print("Number of documents:", len(documents))
print("\nExample title:")
print(talks.loc[0, "title"])

print("\nFirst 500 characters of combined text:")
print(documents[0][:500])

Number of documents: 2550

Example title:
do schools kill creativity?

First 500 characters of combined text:
do schools kill creativity? children creativity culture dance education parenting teaching sir ken robinson makes an entertaining and profoundly moving case for creating an education system that nurtures creativity. good morning. how are you? it's been great, hasn't it? i've been blown away by the whole thing. in fact, i'm leaving. there have been three themes running through the conference which are relevant to what i want to talk about. one is the extraordinary evidence of human creativity in 


In [10]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of documents:", tfidf_matrix.shape[0])
print("Number of vocabulary terms:", tfidf_matrix.shape[1])

TF-IDF matrix shape: (2550, 10000)
Number of documents: 2550
Number of vocabulary terms: 10000


In [11]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("First 30 vocabulary terms:")
print(feature_names[:30])

print("\nTotal vocabulary terms:", len(feature_names))

First 30 vocabulary terms:
['aaron' 'ab' 'abandon' 'abandoned' 'abc' 'abed' 'abilities' 'ability'
 'able' 'abnormal' 'abortion' 'abraham' 'abroad' 'absence' 'absent'
 'absolute' 'absolutely' 'absorb' 'absorbed' 'absorbing' 'abstract'
 'abstraction' 'absurd' 'abu' 'abundance' 'abundant' 'abuse' 'abused'
 'abuses' 'abusive']

Total vocabulary terms: 10000


In [12]:
# Select the first TED Talk
talk_index = 0

# Calculate similarity with all TED Talks
similarity_scores = cosine_similarity(
    tfidf_matrix[talk_index],
    tfidf_matrix
).flatten()

print("Selected TED Talk:")
print(talks.loc[talk_index, "title"])

print("\nSimilarity scores shape:", similarity_scores.shape)

Selected TED Talk:
do schools kill creativity?

Similarity scores shape: (2550,)


In [13]:
similarity_df = pd.DataFrame({
    "title": talks["title"],
    "similarity_score": similarity_scores
})

similarity_df = similarity_df.sort_values(
    by="similarity_score",
    ascending=False
)

display(similarity_df.head(10))

,title,similarity_score
0,do schools kill creativity?,1.000000
692,bring on the learning revolution!,0.386260
1502,how to escape education's death valley,0.385157
2049,how i stopped the taliban from shutting down m...,0.307415
1913,how to run a company with no rules,0.306244
1500,our failing schools. enough is enough!,0.305831
624,my 5 lives as an artist,0.291469
1445,build a school in the cloud,0.289969
715,education innovation in the slums,0.288829
1420,kids need structure,0.287167


In [14]:
def recommend_talks(talk_index, top_n=5):

    # Calculate similarity with all talks
    scores = cosine_similarity(
        tfidf_matrix[talk_index],
        tfidf_matrix
    )[0]

    # Exclude the selected talk
    scores[talk_index] = -1

    # Get indices of top similar talks
    top_indices = np.argsort(scores)[::-1][:top_n]

    # Get recommendation details
    recommendations = talks.iloc[top_indices][
        ["title", "main_speaker", "url"]
    ].copy()

    # Add similarity scores
    recommendations["similarity_score"] = scores[top_indices]

    return recommendations.reset_index(drop=True)

In [15]:
selected_index = 0

print("Selected talk:")
print(talks.loc[selected_index, "title"])

print("\nTop 5 recommended talks:")

display(
    recommend_talks(selected_index, top_n=5)
)

Selected talk:
do schools kill creativity?

Top 5 recommended talks:


,title,main_speaker,url,similarity_score
0,bring on the learning revolution!,Ken Robinson,https://www.ted.com/talks/sir_ken_robinson_bri...,0.386260
1,how to escape education's death valley,Ken Robinson,https://www.ted.com/talks/ken_robinson_how_to_...,0.385157
2,how i stopped the taliban from shutting down m...,Sakena Yacoobi,https://www.ted.com/talks/sakena_yacoobi_how_i...,0.307415
3,how to run a company with no rules,Ricardo Semler,https://www.ted.com/talks/ricardo_semler_how_t...,0.306244
4,our failing schools. enough is enough!,Geoffrey Canada,https://www.ted.com/talks/geoffrey_canada_our_...,0.305831


In [17]:
# Get unique talks with non-empty titles
unique_talks = talks[talks["title"].str.strip() != ""].drop_duplicates("title")

# Select three talks from different positions
selected_talks = unique_talks.iloc[
    np.linspace(0, len(unique_talks) - 1, 3, dtype=int)
]

# Display selected talks
display(selected_talks[["title", "main_speaker", "url"]])

,title,main_speaker,url
0,do schools kill creativity?,Ken Robinson,https://www.ted.com/talks/ken_robinson_says_sc...
1274,four principles for the open world,Don Tapscott,https://www.ted.com/talks/don_tapscott_four_pr...
2549,how a video game might help us build better ci...,Karoliina Korppoo,https://www.ted.com/talks/karoliina_korppoo_ho...


In [18]:
all_recommendations = []

for idx in selected_talks.index:

    result = recommend_talks(idx, top_n=5)

    result["selected_talk"] = talks.loc[idx, "title"]

    all_recommendations.append(result)

recommendation_results = pd.concat(all_recommendations, ignore_index=True)

display(recommendation_results)

,title,main_speaker,url,similarity_score,selected_talk
0,bring on the learning revolution!,Ken Robinson,https://www.ted.com/talks/sir_ken_robinson_bri...,0.386260,do schools kill creativity?
1,how to escape education's death valley,Ken Robinson,https://www.ted.com/talks/ken_robinson_how_to_...,0.385157,do schools kill creativity?
2,how i stopped the taliban from shutting down m...,Sakena Yacoobi,https://www.ted.com/talks/sakena_yacoobi_how_i...,0.307415,do schools kill creativity?
3,how to run a company with no rules,Ricardo Semler,https://www.ted.com/talks/ricardo_semler_how_t...,0.306244,do schools kill creativity?
4,our failing schools. enough is enough!,Geoffrey Canada,https://www.ted.com/talks/geoffrey_canada_our_...,0.305831,do schools kill creativity?
5,demand a more open-source government,Beth Noveck,https://www.ted.com/talks/beth_noveck_demand_a...,0.279678,four principles for the open world
6,how the internet will transform government,Clay Shirky,https://www.ted.com/talks/clay_shirky_how_the_...,0.273364,four principles for the open world
7,the new open-source economics,Yochai Benkler,https://www.ted.com/talks/yochai_benkler_on_th...,0.263172,four principles for the open world
8,our failing schools. enough is enough!,Geoffrey Canada,https://www.ted.com/talks/geoffrey_canada_our_...,0.240913,four principles for the open world
9,the birth of the open-source learning revolution,Richard Baraniuk,https://www.ted.com/talks/richard_baraniuk_on_...,0.237882,four principles for the open world


## Similarity Analysis: Observations

1. Some recommended talks share topics or terminology with the selected talk.
2. Talks with similar words may receive high similarity scores even if their
   overall meaning is different.
3. Talks discussing related ideas with different vocabulary may receive lower
   similarity scores.
4. The selected talk is excluded from its own recommendations because its
   similarity with itself is 1.0.
5. Cosine similarity measures text similarity; it does not guarantee that a
   user will like a recommendation.